# Testing Linear Layer of myNN

In [5]:
import sys
import os

# Add parent directory to path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

print(f"Added '{parent_dir}' to sys.path")

Added 'c:\Users\ahris\Coding\zero_to_cnn' to sys.path


In [8]:
import numpy as np
from myNN import *

print("Successfully imported myNN!")

Successfully imported myNN!


## Test 1: Basic Linear Layer

In [9]:
# Create sample data
X = np.random.rand(10, 3)
Y = np.random.randint(0, 2, size=(10,))  # Classification labels

# Create linear layer
linear = nn.Linear(3, 2)

print("Input shape:", X.shape)
print("Labels shape:", Y.shape)
print("\nWeights shape:", linear.weights.shape)
print("Bias shape:", linear.bias.shape)

Input shape: (10, 3)
Labels shape: (10,)

Weights shape: (3, 2)
Bias shape: (1, 2)


In [10]:
# Forward pass
output = linear(X)
print("Output shape:", output.shape)
print("Output:\n", output)

Output shape: (10, 2)
Output:
 [[ 0.48010881 -0.43923047]
 [ 1.55710887  0.91605086]
 [ 0.43414269 -0.09724236]
 [ 1.13361005  0.71703564]
 [ 1.86809077  0.75059023]
 [ 1.10523151  0.0537487 ]
 [ 0.02339188 -0.06249297]
 [ 0.6286001   0.60969328]
 [ 0.47849169 -0.14567184]
 [ 1.86339065  0.14625313]]


## Test 2: Multi-Layer Network with Backpropagation

In [21]:
# Create network layers
li1 = nn.Linear(3, 5)
relu = AF.ReLU()
li2 = nn.Linear(5, 2)

# Forward pass
print("=== Forward Pass ===")
out1 = li1(X)
print(f"Layer 1 output shape: {out1.shape}")

out2 = relu(out1)
print(f"ReLU output shape: {out2.shape}")

out3 = li2(out2)
print(f"Layer 2 output shape: {out3.shape}")

# Compute loss
loss_fn = LF.CrossEntropyLoss()
loss = loss_fn.calculate(out3, Y)
print(f"\nLoss: {loss:.4f}")

=== Forward Pass ===
Layer 1 output shape: (10, 5)
ReLU output shape: (10, 5)
Layer 2 output shape: (10, 2)

Loss: 0.6883


## Test 3: Backward Pass

In [23]:
print("=== Manual Backward Pass ===")

# Compute gradients for the output layer
# For simplicity, we'll use a simple gradient
batch_size = X.shape[0]
num_classes = 2

# Create one-hot encoded labels
one_hot_labels = np.zeros((batch_size, num_classes))
one_hot_labels[np.arange(batch_size), Y] = 1

# Simple gradient for demonstration: output - labels
dout3 = (out3 - one_hot_labels) / batch_size

print(f"Gradient at output shape: {dout3.shape}")

# Backward through li2
dout2 = li2.backward(dout3)
print(f"Gradient after li2 backward: {dout2.shape}")
print(f"li2.dweights shape: {li2.dweights.shape}")
print(f"li2.dbiases shape: {li2.dbiases.shape}")

# Backward through ReLU
dout1 = relu.backward(dout2)
print(f"Gradient after ReLU backward: {dout1.shape}")

# Backward through li1
dout0 = li1.backward(dout1)
print(f"Gradient after li1 backward: {dout0.shape}")
print(f"li1.dweights shape: {li1.dweights.shape}")
print(f"li1.dbiases shape: {li1.dbiases.shape}")

=== Manual Backward Pass ===
Gradient at output shape: (10, 2)
Gradient after li2 backward: (10, 5)
li2.dweights shape: (5, 2)
li2.dbiases shape: (1, 2)
Gradient after ReLU backward: (10, 5)
Gradient after li1 backward: (10, 3)
li1.dweights shape: (3, 5)
li1.dbiases shape: (1, 5)


## Test 4: Optimizer Step

In [25]:
print("=== Optimizer Test ===")

# Save old weights
old_weights_li1 = li1.weights.copy()
old_weights_li2 = li2.weights.copy()

# Create optimizer and step
optimizer = optim.SGD([li1, li2], lr=0.01)
optimizer.step()

# Check if weights changed
weight_change_li1 = np.mean(np.abs(li1.weights - old_weights_li1))
weight_change_li2 = np.mean(np.abs(li2.weights - old_weights_li2))

print(f"Average weight change in li1: {weight_change_li1:.6f}")
print(f"Average weight change in li2: {weight_change_li2:.6f}")

if weight_change_li1 > 0 and weight_change_li2 > 0:
    print("\n✓ Weights updated successfully!")
else:
    print("\n✗ Warning: Weights did not change!")

=== Optimizer Test ===
Average weight change in li1: 0.003708
Average weight change in li2: 0.001851

✓ Weights updated successfully!


## Test 5: Full Training Loop

In [26]:
print("=== Full Training Loop ===")

# Create fresh network
layer1 = nn.Linear(3, 5)
activation = AF.ReLU()
layer2 = nn.Linear(5, 2)

# Create optimizer and loss function
optimizer = optim.SGD([layer1, layer2], lr=0.1)
loss_fn = LF.CrossEntropyLoss()

# Training loop
num_epochs = 50
for epoch in range(num_epochs):
    # Forward pass
    h1 = layer1(X)
    h2 = activation(h1)
    logits = layer2(h2)
    
    # Compute loss
    loss = loss_fn.calculate(logits, Y)
    
    # Backward pass
    one_hot = np.zeros((batch_size, num_classes))
    one_hot[np.arange(batch_size), Y] = 1
    grad = (logits - one_hot) / batch_size
    
    grad = layer2.backward(grad)
    grad = activation.backward(grad)
    grad = layer1.backward(grad)
    
    # Update weights
    optimizer.step()
    optimizer.zero_grad()
    
    # Print progress
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss:.4f}")

print("\n✓ Training completed!")

=== Full Training Loop ===
Epoch 10/50, Loss: 0.6662
Epoch 20/50, Loss: 0.6483
Epoch 30/50, Loss: 0.6340
Epoch 40/50, Loss: 0.6216
Epoch 50/50, Loss: 0.6106

✓ Training completed!
